# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashizhenya755-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_id), summarized over its trailing 90-day
window, as of the export date the starter CSV was built from. This is a
single-snapshot dataset (not a daily time series), so "the time window" here
means the 90-day lookback baked into fields like impressions_90d and
sessions_90d, not a range of dates I can slice myself.

Verified below: row count, that content_id is unique (confirming one row per
page, not per day or per client), and the columns that carry the 90-day
window in their name.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/yashizhenya755-dev/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [9]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"content_id is unique: {df['content_id'].is_unique}")
print(f"Duplicate content_id rows: {df['content_id'].duplicated().sum()}")

window_cols = [c for c in df.columns if "90d" in c]
print(f"Columns carrying a 90-day window in their name: {window_cols}")

Rows: 30,000
content_id is unique: True
Duplicate content_id rows: 0
Columns carrying a 90-day window in their name: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (safe to use as model input, observed before any decision):
impressions_90d, sessions_90d, clicks_90d, ctr, avg_position,
content_age_days, days_since_last_update, word_count.

Label / proxy: trend_direction (specifically trend_direction == "down") —
this is what the starter model predicts. Flagged as a proxy, not a real
future outcome, per the lane guide.

Context (useful for grouping/joins, not a model feature): content_id,
client_id — these are just identifiers, not signals.

Excluded: any FlyRank product decision flags (health_score, priority_score,
action_type, refresh_tier). Why: the starter dataset doesn't actually ship
these (confirmed below), and even if it did, using a product's own decision
as a model feature or label creates a circular result — the model would
just learn to copy an existing rule instead of finding real signal.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
all_cols = df.columns.tolist()
print(f"All {len(all_cols)} columns:\n{all_cols}")

product_flags = ["health_score", "priority_score", "action_type", "refresh_tier"]
present = [c for c in product_flags if c in df.columns]
print(f"\nProduct decision flags present in this dataset: {present if present else 'none'}")

All 44 columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Product decision flags present in this dataset: none


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Checking: row/column counts, missing values in the fields I plan to use as
features, and the value range of each numeric feature (to catch anything
that looks like a units mismatch, similar to the ctr fraction-vs-percent
issue found earlier).

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = ["impressions_90d", "sessions_90d", "clicks_90d", "ctr",
                "avg_position", "content_age_days", "days_since_last_update",
                "word_count"]

print("Missing values per feature column:")
print(df[feature_cols].isna().sum())

print("\nValue ranges (min / max) per feature column:")
print(df[feature_cols].describe().loc[["min", "max"]])

print(f"\ntrend_direction value counts:\n{df['trend_direction'].value_counts()}")

Missing values per feature column:
impressions_90d              0
sessions_90d                 0
clicks_90d                   0
ctr                          0
avg_position                 0
content_age_days             0
days_since_last_update       0
word_count                7699
dtype: int64

Value ranges (min / max) per feature column:
     impressions_90d  sessions_90d  clicks_90d    ctr  avg_position  \
min              1.0           1.0         0.0    0.0           0.0   
max         517715.0        4345.0      4178.0  100.0         245.0   

     content_age_days  days_since_last_update  word_count  
min              90.0                     1.0         8.0  
max             564.0                   373.0      9546.0  

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This starter dataset can't tell me:

- What happened AFTER this snapshot — there's no future outcome here, only
  the current 90-day window, so any "decline" label is describing the
  present, not predicting the future.
- Anything about individual clients' history length or tracking start
  dates (gsc_data_start, ga4_data_start) — those fields live in the
  warehouse release's dim_clients table, not in this starter CSV, so I
  can't yet check for the "unbalanced panel" issue the lane guide warns
  about.
- Whether a traffic drop is a real decline versus consolidation (a sibling
  page absorbing the traffic) or seasonality — this dataset has no related-
  page or calendar context to check that.
- Anything about real query text, URLs, or client identity — all
  pseudonymized/scrambled by design, so I can't investigate a specific
  page beyond its hashed id and its numeric signals.
- Cause and effect — even if refreshing a page correlates with recovery
  later, this data alone can't prove the refresh caused it.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.